# Pipeline de Detecção de Crises Epilépticas — Notebook 2
## Treinamento dos Modelos (LOSO)

**Dataset:** SeizeIT2 · **Autor:** Danilo Pedro da Silva Valério

---

## 1. Imports

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from xgboost import XGBClassifier
from tqdm.auto import tqdm
import itertools

warnings.filterwarnings('ignore')
print("✅ Imports OK")

✅ Imports OK


c:\Users\danil\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carregar Metadados do Notebook 1

In [2]:
# Ajustar caminho se necessário
DATA_DIR = './data'
meta_path = os.path.join(DATA_DIR, 'pipeline_meta.json')

with open(meta_path, 'r') as f:
    meta = json.load(f)

VALID_SUBJECTS = meta['VALID_SUBJECTS']
WINDOW_CONFIGS = meta['WINDOW_CONFIGS']
SFREQ = meta['SFREQ']
RANDOM_SEED = 42

# Índices de features
level3_fsa = meta['level3_fsa']

print(f"✅ Metadados carregados")
print(f"   Sujeitos válidos: {VALID_SUBJECTS}")
print(f"   Configurações de janela: {list(WINDOW_CONFIGS.keys())}")

✅ Metadados carregados
   Sujeitos válidos: ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012']
   Configurações de janela: ['4s_50', '5s_50', '6s_50']


## 3. Configuração dos Experimentos

**54 experimentos = 3 janelas × 3 FS × 2 ratios × 3 modelos**

### Ratios de undersampling (aplicados no treino e teste):
- **1:3** — 3 non-seizure para cada seizure
- **1:5** — 5 non-seizure para cada seizure

In [3]:
# Configurações experimentais
RUN_CFGS = list(WINDOW_CONFIGS.keys())  # ['4s_50', '5s_50', '6s_50']
RUN_MODELS = ['xgb', 'svm', 'rf']       # XGBoost, SVM, Random Forest
UNDERSAMPLE_RATIOS = [3, 5]             # 1:3 e 1:5

# Mapeamento FS → índice
FS_INDICES = {
    'FS-A': level3_fsa,
}

# Estrutura para armazenar resultados
# full_results[fs_name][cfg_name][ratio][model_key] = (results_list, df)
full_results = {fs: {} for fs in FS_INDICES.keys()}

print("✅ Configuração dos experimentos:")
print(f"   Feature Sets: {list(FS_INDICES.keys())}")
print(f"   Janelas: {RUN_CFGS}")
print(f"   Ratios: {UNDERSAMPLE_RATIOS}")
print(f"   Modelos: {RUN_MODELS}")
print(f"   Total: {len(FS_INDICES) * len(RUN_CFGS) * len(UNDERSAMPLE_RATIOS) * len(RUN_MODELS)} experimentos")

✅ Configuração dos experimentos:
   Feature Sets: ['FS-A']
   Janelas: ['4s_50', '5s_50', '6s_50']
   Ratios: [3, 5]
   Modelos: ['xgb', 'svm', 'rf']
   Total: 18 experimentos


## 4. Funções de Undersampling e Métricas

In [4]:
def apply_ratio(X, y, ratio, seed=RANDOM_SEED):
    """
    Reamostra para ratio non-seizure : seizure.
    Mantém todas as seizure, faz STRIDE nas non-seizure.
    """
    idx_sz  = np.where(y == 1)[0]
    idx_non = np.where(y == 0)[0]
    
    if len(idx_sz) == 0:
        return X[:0], y[:0]
    
    n_want = min(len(idx_non), ratio * len(idx_sz))
    stride = max(1, len(idx_non) // n_want)
    idx_sel = idx_non[::stride][:n_want]
    
    idx_all = np.sort(np.concatenate([idx_sz, idx_sel]))
    return X[idx_all], y[idx_all]


def compute_far(y_test, y_pred, win_sec, overlap):
    """False Alarm Rate por hora (baseado em janelas)."""
    step_sec = win_sec * (1.0 - overlap)
    total_hours = (len(y_test) * step_sec) / 3600.0
    if total_hours == 0:
        return 0.0
    false_positives = ((y_pred == 1) & (y_test == 0)).sum()
    return false_positives / total_hours


def compute_metrics(y_test, y_pred, win_sec, overlap):
    """Calcula métricas básicas."""
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = f1_score(y_test, y_pred, zero_division=0)
    far = compute_far(y_test, y_pred, win_sec, overlap)
    
    return {
        'sensitivity': sensitivity,
        'specificity': specificity,
        'precision': precision,
        'f1': f1,
        'far_per_hour': far,
        'tp': int(tp),
        'fp': int(fp),
        'tn': int(tn),
        'fn': int(fn)
    }

print("✅ Funções auxiliares definidas")

✅ Funções auxiliares definidas


## 5. Função LOSO Genérica

In [5]:
def run_loso(feat_index, subjects, win_sec, overlap, ratio, model_key):
    """
    Leave-One-Subject-Out cross-validation.
    
    Args:
        feat_index: dict {subject: (feat_path, labels_path)}
        subjects: lista de sujeitos válidos
        win_sec: tamanho da janela em segundos
        overlap: fração de overlap (0.5 = 50%)
        ratio: ratio non-seizure:seizure (ex: 3 ou 5) — APLICADO APENAS NO TREINO
        model_key: 'xgb', 'svm', ou 'rf'
    
    Returns:
        Lista de dicionários com métricas por sujeito.
    """
    valid = [s for s in subjects if s in feat_index]
    results = []
    
    for test_sub in valid:
        train_subs = [s for s in valid if s != test_sub]
        
        # Carregar treino
        X_train_list, y_train_list = [], []
        for s in train_subs:
            fp, lp = feat_index[s]
            X_train_list.append(np.load(fp))
            y_train_list.append(np.load(lp))
        
        X_train = np.vstack(X_train_list)
        y_train = np.concatenate(y_train_list)
        
        # Undersampling no treino
        X_train, y_train = apply_ratio(X_train, y_train, ratio)
        
        # Carregar teste
        fp_test, lp_test = feat_index[test_sub]
        X_test = np.load(fp_test)
        y_test = np.load(lp_test)
        
        # Undersampling no teste (aplicado conforme código original)
        X_test, y_test = apply_ratio(X_test, y_test, ratio)
        
        if len(X_train) == 0 or len(X_test) == 0:
            continue
        
        # Normalização
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # Feature selection
        n_features = min(10, X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)
        
        # Modelo
        if model_key == 'xgb':
            model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_SEED,
                eval_metric='logloss'
            )
        elif model_key == 'svm':
            model = SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                random_state=RANDOM_SEED
            )
        elif model_key == 'rf':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_SEED
            )
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Métricas
        metrics = compute_metrics(y_test, y_pred, win_sec, overlap)
        
        # AUC-PR (se modelo suporta predict_proba)
        if hasattr(model, 'predict_proba'):
            y_proba = model.predict_proba(X_test)[:, 1]
            metrics['auc_pr'] = average_precision_score(y_test, y_proba)
        else:
            metrics['auc_pr'] = 0.0
        
        metrics['subject'] = test_sub
        results.append(metrics)
    
    return results

print("✅ Função LOSO definida")

✅ Função LOSO definida


## 6. Executar os Experimentos (BASELINE)

**Loop principal:** percorre todas as combinações de FS × janela × ratio × modelo

In [6]:
# Gerar todas as combinações
combos = list(itertools.product(
    FS_INDICES.keys(),
    RUN_CFGS,
    UNDERSAMPLE_RATIOS,
    RUN_MODELS
))

print(f"\nTotal de experimentos: {len(combos)}")
print("Iniciando loop principal...\n")

for fs_name, cfg_name, ratio, model_key in tqdm(combos, desc='Experimentos', unit='exp'):
    cfg = WINDOW_CONFIGS[cfg_name]
    feat_index = FS_INDICES[fs_name].get(cfg_name, {})
    valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]
    
    if not valid_subs:
        continue
    
    # Criar estrutura de dicionário
    if cfg_name not in full_results[fs_name]:
        full_results[fs_name][cfg_name] = {}
    if ratio not in full_results[fs_name][cfg_name]:
        full_results[fs_name][cfg_name][ratio] = {}
    
    # Executar LOSO
    results = run_loso(
        feat_index=feat_index,
        subjects=valid_subs,
        win_sec=cfg['win_sec'],
        overlap=cfg['overlap'],
        ratio=ratio,
        model_key=model_key
    )
    
    # Converter para DataFrame
    df = pd.DataFrame(results)
    full_results[fs_name][cfg_name][ratio][model_key] = (results, df)

print("\n✅ Todos os experimentos baseline concluídos!")


Total de experimentos: 18
Iniciando loop principal...



Experimentos:   0%|          | 0/18 [00:00<?, ?exp/s]

Experimentos: 100%|██████████| 18/18 [06:02<00:00, 20.13s/exp]


✅ Todos os experimentos baseline concluídos!


## 7. Salvar Resultados Consolidados (BASELINE)

In [7]:
# Consolidar tudo em um único CSV
all_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    results, df = full_results[fs_name][cfg_name][ratio][model_key]
                    for _, row in df.iterrows():
                        row_dict = row.to_dict()
                        row_dict['fs_name'] = fs_name
                        row_dict['cfg_name'] = cfg_name
                        row_dict['ratio'] = ratio
                        row_dict['model'] = model_key
                        all_rows.append(row_dict)
                except KeyError:
                    pass

df_all_baseline = pd.DataFrame(all_rows)
output_path = os.path.join(DATA_DIR, 'results_baseline_all.csv')
df_all_baseline.to_csv(output_path, index=False)

print(f"✅ Resultados baseline salvos em: {output_path}")
print(f"   Total de linhas: {len(df_all_baseline)}")

✅ Resultados baseline salvos em: ./data\results_baseline_all.csv
   Total de linhas: 216


## 8. Tabela de Ranking Final (BASELINE)

Ordenação: **Sensitivity desc → FAR asc**

In [8]:
# Agregar por configuração (média entre sujeitos)
metrics = ['sensitivity', 'specificity', 'f1', 'precision', 'auc_pr', 'far_per_hour']
ranking_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    _, df = full_results[fs_name][cfg_name][ratio][model_key]
                    row = {
                        'FS': fs_name,
                        'Janela': cfg_name,
                        'Ratio': f"1:{ratio}",
                        'Modelo': model_key.upper(),
                    }
                    for m in metrics:
                        row[m] = df[m].mean()
                    ranking_rows.append(row)
                except KeyError:
                    pass

df_ranking_baseline = pd.DataFrame(ranking_rows)
df_ranking_baseline = df_ranking_baseline.sort_values(
    by=['sensitivity', 'far_per_hour'],
    ascending=[False, True]
).reset_index(drop=True)

ranking_path = os.path.join(DATA_DIR, 'ranking_baseline.csv')
df_ranking_baseline.to_csv(ranking_path, index=False)

print("✅ Ranking baseline salvo")
print("\nTop 10 configurações (baseline):")
display(df_ranking_baseline.head(10))

✅ Ranking baseline salvo

Top 10 configurações (baseline):


,FS,Janela,Ratio,Modelo,sensitivity,specificity,f1,precision,auc_pr,far_per_hour
0,FS-A,4s_50,1:3,XGB,0.577975,0.912160,0.599692,0.652904,0.709408,118.584503
1,FS-A,5s_50,1:3,RF,0.573872,0.907891,0.589899,0.643134,0.686957,99.477346
2,FS-A,4s_50,1:3,RF,0.572955,0.908564,0.592539,0.640548,0.697633,123.438281
3,FS-A,6s_50,1:3,RF,0.565755,0.929281,0.598154,0.667350,0.676530,63.646778
4,FS-A,5s_50,1:3,SVM,0.563849,0.921686,0.582876,0.643965,0.000000,84.579567
5,FS-A,5s_50,1:3,XGB,0.563189,0.911596,0.589788,0.658898,0.676099,95.475898
6,FS-A,4s_50,1:3,SVM,0.555706,0.913406,0.582556,0.644951,0.000000,116.902354
7,FS-A,6s_50,1:3,XGB,0.541156,0.918571,0.573482,0.639275,0.676822,73.285781
8,FS-A,6s_50,1:3,SVM,0.529886,0.929796,0.565316,0.640909,0.000000,63.184021
9,FS-A,4s_50,1:5,XGB,0.513092,0.943088,0.542940,0.628329,0.625011,85.368360


---

# ========================================
# PARTE 2 - PIPELINE MELHORADO (FASE 1)
# ========================================

## 🎯 Objetivo
Melhorar o pipeline de detecção com foco em **viabilidade clínica** para EEG wearable de **2 canais** (SeizlT2).

---

## 📋 Mudanças Implementadas

### **1. Undersampling Melhorado (Hard Negative Mining Temporal)**
**Problema anterior:** Stride simples pegava amostras uniformemente distribuídas no tempo.

**Solução nova:**
- Para cada evento de crise, selecionar janelas **não-ictais próximas temporalmente** (±30-60s)
- Força o modelo a distinguir atividade normal próxima vs crise real
- Amostras distantes são "fáceis demais" e não ajudam

**Por quê funciona melhor:**
- Com apenas 2 canais, informação espacial é limitada
- Contexto temporal se torna crítico
- Hard negatives aumentam discriminação

---

### **2. Uso de Probabilidades (predict_proba)**
**Problema anterior:** `model.predict()` retorna decisões binárias (0/1).

**Solução nova:**
- Usar `model.predict_proba(X_test)[:, 1]` para obter probabilidades calibradas
- Permite threshold tuning para balancear sensitivity/FAR
- Habilita pós-processamento baseado em confiança

---

### **3. Pipeline de Pós-Processamento Temporal (ESSENCIAL)**
**Ordem de aplicação:**
```
probs → threshold → histerese → k_consecutivas → conversão para eventos → merge → min_duration → métricas
```

**Etapas:**

**a) Histerese:**
- `th_on = 0.5`: probabilidade para ATIVAR alarme
- `th_off = 0.3`: probabilidade para DESATIVAR alarme
- Evita oscilações rápidas (liga-desliga-liga)

**b) Filtro de k janelas consecutivas:**
- `k = 3`: requer 3 janelas consecutivas positivas
- Remove spikes isolados (ruído)

**c) Conversão janelas → eventos:**
- Identifica segmentos contínuos de detecção
- Retorna (start_time, end_time) para cada evento

**d) Merge de eventos próximos:**
- `max_gap = 10s`: se dois eventos estão a <10s, unir
- Evita contar múltiplas detecções da mesma crise

**e) Remoção de eventos curtos:**
- `min_duration = 5s`: descartar eventos <5s
- Crises reais duram mais que alguns segundos

---

### **4. Métricas Corretas (Baseadas em Eventos)**
**Problema anterior:** FAR calculado por janelas (inflado artificialmente).

**Solução nova:**
- **Sensitivity**: proporção de crises reais detectadas
- **FAR**: falsos alarmes POR HORA (não por janela)
- **AUC-PR**: `average_precision_score(y_true, probs)`
- **Event-level matching**: detecção conta se overlap ≥50% com crise real

---

### **5. NÃO Aplicar Ratio no Teste**
**Problema anterior:** Ratio aplicado no teste distorce FAR.

**Solução nova:**
- **Treino**: usar ratio 1:3 ou 1:5 (balanceamento)
- **Teste**: usar TODAS as janelas (sem subsampling)
- FAR agora reflete cenário real de uso

---

### **6. Threshold Tuning**
Testar múltiplos thresholds: `[0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]`

**Critério de escolha:**
- Sensitivity ≥ 0.6 (detectar pelo menos 60% das crises)
- Menor FAR possível

---

## 🔬 Resultados Esperados
Com FASE 1 implementada corretamente:
- ✅ FAR: de ~40-80/h → **10-25/h**
- ✅ Sensitivity: ~0.5-0.6 (mais estável)
- ✅ Precision: melhora significativa
- ✅ AUC-PR: melhora significativa

---

## 📊 Estrutura do Código V2
- `apply_ratio_v2()`: undersampling temporal
- `apply_hysteresis()`: histerese em probabilidades
- `apply_k_consecutive()`: filtro de k janelas
- `probs_to_events()`: conversão para eventos
- `merge_events()`: união de eventos próximos
- `filter_short_events()`: remoção de eventos curtos
- `compute_metrics_v2()`: métricas baseadas em eventos
- `run_loso_v2()`: função LOSO melhorada

**Variáveis:**
- `full_results_v2`: resultados da versão melhorada
- `df_all_v2`, `df_ranking_v2`: dataframes comparativos

---

## 9. Funções de Undersampling Melhorado (V2)

In [9]:
def apply_ratio_v2(X, y, ratio, win_sec, overlap, seed=RANDOM_SEED):
    """
    Hard negative mining temporal.
    
    Para cada evento de crise:
    - Mantém todas as janelas ictais
    - Seleciona janelas não-ictais em janela temporal de ±60s ao redor
    - Completa com amostras aleatórias se necessário
    
    Args:
        X: features [n_windows, n_features]
        y: labels [n_windows]
        ratio: quantas non-seizure para cada seizure
        win_sec: tamanho da janela em segundos
        overlap: fração de overlap (0.5 = 50%)
        seed: random seed
    
    Returns:
        X_sampled, y_sampled: arrays reamostrados
    """
    np.random.seed(seed)
    
    idx_sz = np.where(y == 1)[0]
    idx_non = np.where(y == 0)[0]
    
    if len(idx_sz) == 0:
        return X[:0], y[:0]
    
    # Passo 1: Identificar eventos de crise (sequências contínuas)
    seizure_events = []
    current_event = [idx_sz[0]]
    
    for i in range(1, len(idx_sz)):
        if idx_sz[i] == idx_sz[i-1] + 1:
            current_event.append(idx_sz[i])
        else:
            seizure_events.append(current_event)
            current_event = [idx_sz[i]]
    seizure_events.append(current_event)
    
    # Passo 2: Para cada evento, selecionar hard negatives
    step_sec = win_sec * (1.0 - overlap)
    context_window_sec = 60.0  # ±60s ao redor da crise
    context_window_idx = int(context_window_sec / step_sec)
    
    hard_negatives = set()
    
    for event_indices in seizure_events:
        event_start = event_indices[0]
        event_end = event_indices[-1]
        
        # Janela antes da crise
        before_start = max(0, event_start - context_window_idx)
        before_end = event_start
        
        # Janela depois da crise
        after_start = event_end + 1
        after_end = min(len(y), event_end + context_window_idx + 1)
        
        # Selecionar não-ictais na janela temporal
        before_candidates = [i for i in range(before_start, before_end) if y[i] == 0]
        after_candidates = [i for i in range(after_start, after_end) if y[i] == 0]
        
        hard_negatives.update(before_candidates)
        hard_negatives.update(after_candidates)
    
    hard_negatives = np.array(sorted(hard_negatives))
    
    # Passo 3: Completar com amostras aleatórias se necessário
    n_want = ratio * len(idx_sz)
    
    if len(hard_negatives) >= n_want:
        # Temos hard negatives suficientes, selecionar aleatoriamente
        selected_non = np.random.choice(hard_negatives, size=n_want, replace=False)
    else:
        # Usar todos os hard negatives + complementar com aleatórios
        remaining = [i for i in idx_non if i not in hard_negatives]
        n_remaining = n_want - len(hard_negatives)
        
        if len(remaining) >= n_remaining:
            extra = np.random.choice(remaining, size=n_remaining, replace=False)
        else:
            extra = np.array(remaining)
        
        selected_non = np.concatenate([hard_negatives, extra])
    
    # Passo 4: Combinar seizure + non-seizure
    idx_all = np.sort(np.concatenate([idx_sz, selected_non]))
    
    return X[idx_all], y[idx_all]

print("✅ Função apply_ratio_v2 (hard negative mining) definida")

✅ Função apply_ratio_v2 (hard negative mining) definida


## 10. Funções de Pós-Processamento Temporal (V2)

In [10]:
def apply_hysteresis(probs, th_on=0.5, th_off=0.3):
    """
    Aplica histerese em probabilidades.
    
    Estado ON: ativado quando prob >= th_on
    Estado OFF: desativado quando prob < th_off
    Entre th_off e th_on: mantém estado anterior
    
    Args:
        probs: array de probabilidades [n_windows]
        th_on: threshold para ativar
        th_off: threshold para desativar
    
    Returns:
        detections: array binário [n_windows]
    """
    detections = np.zeros(len(probs), dtype=int)
    state = 0  # 0 = OFF, 1 = ON
    
    for i, p in enumerate(probs):
        if state == 0:  # Estado OFF
            if p >= th_on:
                state = 1
                detections[i] = 1
        else:  # Estado ON
            if p < th_off:
                state = 0
                detections[i] = 0
            else:
                detections[i] = 1
    
    return detections


def apply_k_consecutive(detections, k=3):
    """
    Filtro de k janelas consecutivas.
    Uma janela só é positiva se ela E as (k-1) seguintes forem positivas.
    
    Args:
        detections: array binário [n_windows]
        k: número de janelas consecutivas necessárias
    
    Returns:
        filtered: array binário [n_windows]
    """
    filtered = np.zeros(len(detections), dtype=int)
    
    for i in range(len(detections) - k + 1):
        if np.all(detections[i:i+k] == 1):
            filtered[i:i+k] = 1
    
    return filtered


def probs_to_events(detections, win_sec, overlap):
    """
    Converte janelas binárias para eventos (start_time, end_time).
    
    Args:
        detections: array binário [n_windows]
        win_sec: tamanho da janela em segundos
        overlap: fração de overlap (0.5 = 50%)
    
    Returns:
        events: lista de tuplas [(start_sec, end_sec), ...]
    """
    step_sec = win_sec * (1.0 - overlap)
    events = []
    in_event = False
    event_start = 0
    
    for i, det in enumerate(detections):
        if det == 1 and not in_event:
            # Início de evento
            event_start = i * step_sec
            in_event = True
        elif det == 0 and in_event:
            # Fim de evento
            event_end = i * step_sec
            events.append((event_start, event_end))
            in_event = False
    
    # Se terminou em detecção
    if in_event:
        event_end = len(detections) * step_sec
        events.append((event_start, event_end))
    
    return events


def merge_events(events, max_gap=10.0):
    """
    Une eventos próximos se gap < max_gap segundos.
    
    Args:
        events: lista de tuplas [(start, end), ...]
        max_gap: gap máximo em segundos
    
    Returns:
        merged: lista de tuplas [(start, end), ...]
    """
    if not events:
        return []
    
    events = sorted(events, key=lambda x: x[0])
    merged = [events[0]]
    
    for current in events[1:]:
        last_start, last_end = merged[-1]
        curr_start, curr_end = current
        
        if curr_start - last_end <= max_gap:
            # Merge
            merged[-1] = (last_start, max(last_end, curr_end))
        else:
            merged.append(current)
    
    return merged


def filter_short_events(events, min_duration=5.0):
    """
    Remove eventos com duração < min_duration.
    
    Args:
        events: lista de tuplas [(start, end), ...]
        min_duration: duração mínima em segundos
    
    Returns:
        filtered: lista de tuplas [(start, end), ...]
    """
    return [(s, e) for s, e in events if (e - s) >= min_duration]

print("✅ Funções de pós-processamento temporal definidas")

✅ Funções de pós-processamento temporal definidas


## 11. Funções de Métricas Melhoradas (V2)

In [11]:
def compute_metrics_v2(y_test, probs, win_sec, overlap, 
                       th_on=0.5, th_off=0.3, k=3, 
                       max_gap=10.0, min_duration=5.0):
    """
    Calcula métricas com pipeline completo de pós-processamento.
    
    Pipeline:
    probs → histerese → k_consecutivas → eventos → merge → min_duration → métricas
    
    Args:
        y_test: labels verdadeiros [n_windows]
        probs: probabilidades do modelo [n_windows]
        win_sec: tamanho da janela em segundos
        overlap: fração de overlap
        th_on: threshold de ativação para histerese
        th_off: threshold de desativação para histerese
        k: número de janelas consecutivas
        max_gap: gap máximo para merge (segundos)
        min_duration: duração mínima de evento (segundos)
    
    Returns:
        dict com métricas
    """
    # Etapa 1: Histerese
    detections = apply_hysteresis(probs, th_on=th_on, th_off=th_off)
    
    # Etapa 2: K-consecutivas
    detections = apply_k_consecutive(detections, k=k)
    
    # Etapa 3: Conversão para eventos
    pred_events = probs_to_events(detections, win_sec, overlap)
    
    # Etapa 4: Merge de eventos próximos
    pred_events = merge_events(pred_events, max_gap=max_gap)
    
    # Etapa 5: Filtrar eventos curtos
    pred_events = filter_short_events(pred_events, min_duration=min_duration)
    
    # Converter ground truth para eventos
    true_events = probs_to_events(y_test, win_sec, overlap)
    
    # Calcular sensitivity (detecção de crises reais)
    n_true_seizures = len(true_events)
    detected_seizures = 0
    
    for true_start, true_end in true_events:
        for pred_start, pred_end in pred_events:
            # Overlap
            overlap_start = max(true_start, pred_start)
            overlap_end = min(true_end, pred_end)
            overlap_duration = max(0, overlap_end - overlap_start)
            
            true_duration = true_end - true_start
            
            # Considerar detectado se overlap >= 50% da crise real
            if overlap_duration >= 0.5 * true_duration:
                detected_seizures += 1
                break
    
    sensitivity = detected_seizures / n_true_seizures if n_true_seizures > 0 else 0.0
    
    # Calcular FAR (falsos alarmes por hora)
    step_sec = win_sec * (1.0 - overlap)
    total_hours = (len(y_test) * step_sec) / 3600.0
    
    # Falsos alarmes = eventos preditos que NÃO tem overlap com crises reais
    false_alarms = 0
    
    for pred_start, pred_end in pred_events:
        is_false_alarm = True
        
        for true_start, true_end in true_events:
            overlap_start = max(true_start, pred_start)
            overlap_end = min(true_end, pred_end)
            
            if overlap_end > overlap_start:  # Há overlap
                is_false_alarm = False
                break
        
        if is_false_alarm:
            false_alarms += 1
    
    far = false_alarms / total_hours if total_hours > 0 else 0.0
    
    # AUC-PR
    auc_pr = average_precision_score(y_test, probs) if len(np.unique(y_test)) > 1 else 0.0
    
    # Precision baseada em eventos
    n_pred_events = len(pred_events)
    precision = (n_pred_events - false_alarms) / n_pred_events if n_pred_events > 0 else 0.0
    
    return {
        'sensitivity': sensitivity,
        'far_per_hour': far,
        'precision': precision,
        'auc_pr': auc_pr,
        'n_true_seizures': n_true_seizures,
        'detected_seizures': detected_seizures,
        'n_pred_events': n_pred_events,
        'false_alarms': false_alarms,
        'total_hours': total_hours
    }

print("✅ Função compute_metrics_v2 (baseada em eventos) definida")

✅ Função compute_metrics_v2 (baseada em eventos) definida


## 12. Função LOSO V2 (Pipeline Completo Melhorado)

In [12]:
def run_loso_v2(feat_index, subjects, win_sec, overlap, ratio, model_key,
                th_on=0.5, th_off=0.3, k=3, max_gap=10.0, min_duration=5.0):
    """
    Leave-One-Subject-Out cross-validation - VERSÃO MELHORADA.
    
    Mudanças principais:
    1. Undersampling melhorado (hard negative mining temporal) - APENAS NO TREINO
    2. Usa predict_proba em vez de predict
    3. NÃO aplica ratio no teste (usa TODAS as janelas)
    4. Pipeline de pós-processamento temporal completo
    5. Métricas baseadas em eventos (não janelas)
    
    Args:
        feat_index: dict {subject: (feat_path, labels_path)}
        subjects: lista de sujeitos válidos
        win_sec: tamanho da janela em segundos
        overlap: fração de overlap (0.5 = 50%)
        ratio: ratio non-seizure:seizure (ex: 3 ou 5) — APLICADO APENAS NO TREINO
        model_key: 'xgb', 'svm', ou 'rf'
        th_on: threshold de ativação (histerese)
        th_off: threshold de desativação (histerese)
        k: número de janelas consecutivas
        max_gap: gap máximo para merge de eventos (segundos)
        min_duration: duração mínima de evento (segundos)
    
    Returns:
        Lista de dicionários com métricas por sujeito.
    """
    valid = [s for s in subjects if s in feat_index]
    results = []
    
    for test_sub in valid:
        train_subs = [s for s in valid if s != test_sub]
        
        # Carregar treino
        X_train_list, y_train_list = [], []
        for s in train_subs:
            fp, lp = feat_index[s]
            X_train_list.append(np.load(fp))
            y_train_list.append(np.load(lp))
        
        X_train = np.vstack(X_train_list)
        y_train = np.concatenate(y_train_list)
        
        # MUDANÇA 1: Undersampling MELHORADO no treino (hard negative mining)
        X_train, y_train = apply_ratio_v2(X_train, y_train, ratio, win_sec, overlap)
        
        # Carregar teste
        fp_test, lp_test = feat_index[test_sub]
        X_test = np.load(fp_test)
        y_test = np.load(lp_test)
        
        # MUDANÇA 2: NÃO aplicar ratio no teste (usar TODAS as janelas)
        # X_test, y_test permanecem intactos
        
        if len(X_train) == 0 or len(X_test) == 0:
            continue
        
        # Normalização
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # Feature selection
        n_features = min(10, X_train.shape[1])
        selector = SelectKBest(mutual_info_classif, k=n_features)
        X_train = selector.fit_transform(X_train, y_train)
        X_test = selector.transform(X_test)
        
        # Modelo
        if model_key == 'xgb':
            model = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=RANDOM_SEED,
                eval_metric='logloss'
            )
        elif model_key == 'svm':
            # SVM com probabilidades
            model = SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                probability=True,  # IMPORTANTE: habilitar probabilidades
                random_state=RANDOM_SEED
            )
        elif model_key == 'rf':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=RANDOM_SEED
            )
        
        model.fit(X_train, y_train)
        
        # MUDANÇA 3: Usar predict_proba em vez de predict
        probs = model.predict_proba(X_test)[:, 1]
        
        # MUDANÇA 4: Métricas com pós-processamento temporal completo
        metrics = compute_metrics_v2(
            y_test=y_test,
            probs=probs,
            win_sec=win_sec,
            overlap=overlap,
            th_on=th_on,
            th_off=th_off,
            k=k,
            max_gap=max_gap,
            min_duration=min_duration
        )
        
        metrics['subject'] = test_sub
        results.append(metrics)
    
    return results

print("✅ Função run_loso_v2 (pipeline completo melhorado) definida")

✅ Função run_loso_v2 (pipeline completo melhorado) definida


## 13. Executar Experimentos V2 (Pipeline Melhorado)

In [13]:
# Estrutura para armazenar resultados V2
full_results_v2 = {fs: {} for fs in FS_INDICES.keys()}

# Parâmetros de pós-processamento (ajustar se necessário)
PARAMS_V2 = {
    'th_on': 0.5,         # threshold de ativação
    'th_off': 0.3,        # threshold de desativação
    'k': 3,               # k janelas consecutivas
    'max_gap': 10.0,      # merge eventos com gap < 10s
    'min_duration': 5.0   # remover eventos < 5s
}

print("\n" + "="*60)
print("EXECUTANDO PIPELINE MELHORADO (V2)")
print("="*60)
print(f"\nParâmetros de pós-processamento:")
for k, v in PARAMS_V2.items():
    print(f"  {k}: {v}")

print(f"\nTotal de experimentos: {len(combos)}")
print("Iniciando loop V2...\n")

for fs_name, cfg_name, ratio, model_key in tqdm(combos, desc='Experimentos V2', unit='exp'):
    cfg = WINDOW_CONFIGS[cfg_name]
    feat_index = FS_INDICES[fs_name].get(cfg_name, {})
    valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]
    
    if not valid_subs:
        continue
    
    # Criar estrutura de dicionário
    if cfg_name not in full_results_v2[fs_name]:
        full_results_v2[fs_name][cfg_name] = {}
    if ratio not in full_results_v2[fs_name][cfg_name]:
        full_results_v2[fs_name][cfg_name][ratio] = {}
    
    # Executar LOSO V2
    results_v2 = run_loso_v2(
        feat_index=feat_index,
        subjects=valid_subs,
        win_sec=cfg['win_sec'],
        overlap=cfg['overlap'],
        ratio=ratio,
        model_key=model_key,
        **PARAMS_V2
    )
    
    # Converter para DataFrame
    df_v2 = pd.DataFrame(results_v2)
    full_results_v2[fs_name][cfg_name][ratio][model_key] = (results_v2, df_v2)

print("\n✅ Todos os experimentos V2 concluídos!")


EXECUTANDO PIPELINE MELHORADO (V2)

Parâmetros de pós-processamento:
  th_on: 0.5
  th_off: 0.3
  k: 3
  max_gap: 10.0
  min_duration: 5.0

Total de experimentos: 18
Iniciando loop V2...



Experimentos V2: 100%|██████████| 18/18 [29:54<00:00, 99.69s/exp] 


✅ Todos os experimentos V2 concluídos!


## 14. Salvar Resultados V2

In [14]:
# Consolidar tudo em um único CSV
all_rows_v2 = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    results_v2, df_v2 = full_results_v2[fs_name][cfg_name][ratio][model_key]
                    for _, row in df_v2.iterrows():
                        row_dict = row.to_dict()
                        row_dict['fs_name'] = fs_name
                        row_dict['cfg_name'] = cfg_name
                        row_dict['ratio'] = ratio
                        row_dict['model'] = model_key
                        all_rows_v2.append(row_dict)
                except KeyError:
                    pass

df_all_v2 = pd.DataFrame(all_rows_v2)
output_path_v2 = os.path.join(DATA_DIR, 'results_v2_all.csv')
df_all_v2.to_csv(output_path_v2, index=False)

print(f"✅ Resultados V2 salvos em: {output_path_v2}")
print(f"   Total de linhas: {len(df_all_v2)}")

✅ Resultados V2 salvos em: ./data\results_v2_all.csv
   Total de linhas: 216


## 15. Ranking V2

In [15]:
# Agregar por configuração (média entre sujeitos)
metrics_v2 = ['sensitivity', 'precision', 'auc_pr', 'far_per_hour']
ranking_rows_v2 = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    _, df_v2 = full_results_v2[fs_name][cfg_name][ratio][model_key]
                    row = {
                        'FS': fs_name,
                        'Janela': cfg_name,
                        'Ratio': f"1:{ratio}",
                        'Modelo': model_key.upper(),
                    }
                    for m in metrics_v2:
                        row[m] = df_v2[m].mean()
                    ranking_rows_v2.append(row)
                except KeyError:
                    pass

df_ranking_v2 = pd.DataFrame(ranking_rows_v2)
df_ranking_v2 = df_ranking_v2.sort_values(
    by=['sensitivity', 'far_per_hour'],
    ascending=[False, True]
).reset_index(drop=True)

ranking_path_v2 = os.path.join(DATA_DIR, 'ranking_v2.csv')
df_ranking_v2.to_csv(ranking_path_v2, index=False)

print("✅ Ranking V2 salvo")
print("\nTop 10 configurações (V2):")
display(df_ranking_v2.head(10))

✅ Ranking V2 salvo

Top 10 configurações (V2):


,FS,Janela,Ratio,Modelo,sensitivity,precision,auc_pr,far_per_hour
0,FS-A,6s_50,1:3,RF,0.626389,0.032994,0.160055,9.748436
1,FS-A,4s_50,1:3,XGB,0.615278,0.024029,0.119530,12.253852
2,FS-A,6s_50,1:3,SVM,0.558333,0.033171,0.149595,7.884449
3,FS-A,4s_50,1:3,RF,0.547222,0.023851,0.137208,12.683698
4,FS-A,5s_50,1:3,SVM,0.537500,0.029667,0.132167,8.998611
5,FS-A,5s_50,1:3,XGB,0.531944,0.036812,0.127555,10.863137
6,FS-A,5s_50,1:5,SVM,0.511111,0.073511,0.136413,6.138378
7,FS-A,4s_50,1:3,SVM,0.511111,0.021567,0.120649,10.837767
8,FS-A,5s_50,1:3,RF,0.511111,0.033029,0.148714,10.974820
9,FS-A,6s_50,1:3,XGB,0.501389,0.037383,0.135811,9.786169


## 16. Comparação Baseline vs V2

In [16]:
print("\n" + "="*80)
print("COMPARAÇÃO: BASELINE vs V2 (Pipeline Melhorado)")
print("="*80)

# Métricas comuns
common_metrics = ['sensitivity', 'precision', 'auc_pr', 'far_per_hour']

# Criar tabela de comparação
comparison_rows = []

for fs_name in FS_INDICES.keys():
    for cfg_name in RUN_CFGS:
        for ratio in UNDERSAMPLE_RATIOS:
            for model_key in RUN_MODELS:
                try:
                    # Baseline
                    _, df_baseline = full_results[fs_name][cfg_name][ratio][model_key]
                    
                    # V2
                    _, df_v2 = full_results_v2[fs_name][cfg_name][ratio][model_key]
                    
                    row = {
                        'FS': fs_name,
                        'Janela': cfg_name,
                        'Ratio': f"1:{ratio}",
                        'Modelo': model_key.upper(),
                    }
                    
                    for m in common_metrics:
                        baseline_val = df_baseline[m].mean() if m in df_baseline else 0
                        v2_val = df_v2[m].mean() if m in df_v2 else 0
                        
                        row[f'{m}_baseline'] = baseline_val
                        row[f'{m}_v2'] = v2_val
                        
                        # Melhoria percentual
                        if m == 'far_per_hour':
                            # Para FAR, redução é melhoria
                            improvement = ((baseline_val - v2_val) / baseline_val * 100) if baseline_val > 0 else 0
                        else:
                            # Para outros, aumento é melhoria
                            improvement = ((v2_val - baseline_val) / baseline_val * 100) if baseline_val > 0 else 0
                        
                        row[f'{m}_improvement_%'] = improvement
                    
                    comparison_rows.append(row)
                    
                except KeyError:
                    pass

df_comparison = pd.DataFrame(comparison_rows)

# Salvar comparação
comparison_path = os.path.join(DATA_DIR, 'comparison_baseline_vs_v2.csv')
df_comparison.to_csv(comparison_path, index=False)

print(f"\n✅ Comparação salva em: {comparison_path}")

# Mostrar estatísticas gerais
print("\n" + "="*80)
print("ESTATÍSTICAS GERAIS DE MELHORIA")
print("="*80)

for m in common_metrics:
    improvement_col = f'{m}_improvement_%'
    if improvement_col in df_comparison:
        avg_improvement = df_comparison[improvement_col].mean()
        metric_label = m.replace('_', ' ').title()
        
        if m == 'far_per_hour':
            print(f"\n{metric_label}: {avg_improvement:+.1f}% (redução média)")
        else:
            print(f"\n{metric_label}: {avg_improvement:+.1f}% (melhoria média)")

# Melhor configuração V2
print("\n" + "="*80)
print("MELHOR CONFIGURAÇÃO (V2)")
print("="*80)

best_idx = df_ranking_v2.iloc[0]
print(f"\nFS: {best_idx['FS']}")
print(f"Janela: {best_idx['Janela']}")
print(f"Ratio: {best_idx['Ratio']}")
print(f"Modelo: {best_idx['Modelo']}")
print(f"\nMétricas:")
print(f"  Sensitivity: {best_idx['sensitivity']:.3f}")
print(f"  Precision: {best_idx['precision']:.3f}")
print(f"  AUC-PR: {best_idx['auc_pr']:.3f}")
print(f"  FAR: {best_idx['far_per_hour']:.2f} eventos/hora")

print("\n" + "="*80)
print("✅ COMPARAÇÃO CONCLUÍDA!")
print("="*80)


COMPARAÇÃO: BASELINE vs V2 (Pipeline Melhorado)

✅ Comparação salva em: ./data\comparison_baseline_vs_v2.csv

ESTATÍSTICAS GERAIS DE MELHORIA

Sensitivity: -2.5% (melhoria média)

Precision: -92.9% (melhoria média)

Auc Pr: -51.7% (melhoria média)

Far Per Hour: +88.2% (redução média)

MELHOR CONFIGURAÇÃO (V2)

FS: FS-A
Janela: 6s_50
Ratio: 1:3
Modelo: RF

Métricas:
  Sensitivity: 0.626
  Precision: 0.033
  AUC-PR: 0.160
  FAR: 9.75 eventos/hora

✅ COMPARAÇÃO CONCLUÍDA!


## 17. Threshold Tuning (Análise de Sensibilidade)

In [17]:
print("\n" + "="*80)
print("THRESHOLD TUNING - Análise de Sensibilidade")
print("="*80)

# Selecionar melhor configuração para análise
best_config = df_ranking_v2.iloc[0]
fs_name = best_config['FS']
cfg_name = best_config['Janela']
ratio = int(best_config['Ratio'].split(':')[1])
model_key = best_config['Modelo'].lower()

print(f"\nAnalisando configuração:")
print(f"  FS: {fs_name}")
print(f"  Janela: {cfg_name}")
print(f"  Ratio: 1:{ratio}")
print(f"  Modelo: {model_key.upper()}")

# Thresholds para testar
thresholds = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]

print(f"\nTestando {len(thresholds)} thresholds...\n")

# Carregar dados
cfg = WINDOW_CONFIGS[cfg_name]
feat_index = FS_INDICES[fs_name][cfg_name]
valid_subs = [s for s in VALID_SUBJECTS if s in feat_index]

threshold_results = []

for th in tqdm(thresholds, desc='Thresholds'):
    # Executar LOSO com threshold específico
    results_th = run_loso_v2(
        feat_index=feat_index,
        subjects=valid_subs,
        win_sec=cfg['win_sec'],
        overlap=cfg['overlap'],
        ratio=ratio,
        model_key=model_key,
        th_on=th,
        th_off=th * 0.6,  # th_off = 60% de th_on
        k=3,
        max_gap=10.0,
        min_duration=5.0
    )
    
    df_th = pd.DataFrame(results_th)
    
    threshold_results.append({
        'threshold': th,
        'sensitivity': df_th['sensitivity'].mean(),
        'precision': df_th['precision'].mean(),
        'far_per_hour': df_th['far_per_hour'].mean(),
        'auc_pr': df_th['auc_pr'].mean()
    })

df_thresholds = pd.DataFrame(threshold_results)

# Salvar análise de thresholds
threshold_path = os.path.join(DATA_DIR, 'threshold_analysis.csv')
df_thresholds.to_csv(threshold_path, index=False)

print(f"\n✅ Análise de thresholds salva em: {threshold_path}")
print("\nResultados:")
display(df_thresholds)

# Identificar melhor threshold
# Critério: sensitivity >= 0.6 e menor FAR
candidates = df_thresholds[df_thresholds['sensitivity'] >= 0.6]

if len(candidates) > 0:
    best_th = candidates.loc[candidates['far_per_hour'].idxmin()]
    print("\n" + "="*80)
    print("MELHOR THRESHOLD (Sensitivity >= 0.6, menor FAR)")
    print("="*80)
    print(f"\nThreshold: {best_th['threshold']:.2f}")
    print(f"  Sensitivity: {best_th['sensitivity']:.3f}")
    print(f"  Precision: {best_th['precision']:.3f}")
    print(f"  FAR: {best_th['far_per_hour']:.2f} eventos/hora")
    print(f"  AUC-PR: {best_th['auc_pr']:.3f}")
else:
    print("\n⚠️ Nenhum threshold atingiu sensitivity >= 0.6")
    print("Mostrando threshold com maior sensitivity:\n")
    best_th = df_thresholds.loc[df_thresholds['sensitivity'].idxmax()]
    print(f"Threshold: {best_th['threshold']:.2f}")
    print(f"  Sensitivity: {best_th['sensitivity']:.3f}")
    print(f"  FAR: {best_th['far_per_hour']:.2f} eventos/hora")


THRESHOLD TUNING - Análise de Sensibilidade

Analisando configuração:
  FS: FS-A
  Janela: 6s_50
  Ratio: 1:3
  Modelo: RF

Testando 9 thresholds...



Thresholds: 100%|██████████| 9/9 [06:29<00:00, 43.30s/it]


✅ Análise de thresholds salva em: ./data\threshold_analysis.csv

Resultados:


,threshold,sensitivity,precision,far_per_hour,auc_pr
0,0.20,0.805556,0.011691,18.312811,0.160055
1,0.25,0.752778,0.013500,17.275707,0.160055
2,0.30,0.700000,0.016686,15.778820,0.160055
3,0.35,0.688889,0.018921,14.134739,0.160055
4,0.40,0.647222,0.022241,12.587817,0.160055
5,0.45,0.647222,0.027136,11.111833,0.160055
6,0.50,0.626389,0.032994,9.748436,0.160055
7,0.55,0.600000,0.047118,8.426549,0.160055
8,0.60,0.469444,0.091220,7.144048,0.160055



MELHOR THRESHOLD (Sensitivity >= 0.6, menor FAR)

Threshold: 0.55
  Sensitivity: 0.600
  Precision: 0.047
  FAR: 8.43 eventos/hora
  AUC-PR: 0.160


## 18. Sumário Final

In [18]:
print("\n" + "="*80)
print("SUMÁRIO FINAL - BASELINE vs V2")
print("="*80)

# Estatísticas agregadas
print("\n1. MÉTRICAS GERAIS (média de todas as configurações)")
print("-" * 80)

for m in common_metrics:
    baseline_col = f'{m}_baseline'
    v2_col = f'{m}_v2'
    
    if baseline_col in df_comparison and v2_col in df_comparison:
        baseline_avg = df_comparison[baseline_col].mean()
        v2_avg = df_comparison[v2_col].mean()
        
        if m == 'far_per_hour':
            change = baseline_avg - v2_avg
            pct = (change / baseline_avg * 100) if baseline_avg > 0 else 0
            print(f"\n{m.upper()}:")
            print(f"  Baseline: {baseline_avg:.2f}")
            print(f"  V2: {v2_avg:.2f}")
            print(f"  Redução: {change:.2f} ({pct:.1f}%)")
        else:
            change = v2_avg - baseline_avg
            pct = (change / baseline_avg * 100) if baseline_avg > 0 else 0
            print(f"\n{m.upper()}:")
            print(f"  Baseline: {baseline_avg:.3f}")
            print(f"  V2: {v2_avg:.3f}")
            print(f"  Melhoria: {change:+.3f} ({pct:+.1f}%)")

print("\n\n2. TOP 3 CONFIGURAÇÕES - BASELINE")
print("-" * 80)
display(df_ranking_baseline.head(3)[['FS', 'Janela', 'Ratio', 'Modelo', 'sensitivity', 'far_per_hour']])

print("\n2. TOP 3 CONFIGURAÇÕES - V2")
print("-" * 80)
display(df_ranking_v2.head(3)[['FS', 'Janela', 'Ratio', 'Modelo', 'sensitivity', 'far_per_hour']])

print("\n\n3. ARQUIVOS SALVOS")
print("-" * 80)
print(f"  • results_baseline_all.csv")
print(f"  • ranking_baseline.csv")
print(f"  • results_v2_all.csv")
print(f"  • ranking_v2.csv")
print(f"  • comparison_baseline_vs_v2.csv")
print(f"  • threshold_analysis.csv")

print("\n\n4. PRÓXIMOS PASSOS (se FAR ainda > 15/h)")
print("-" * 80)
print("  1. Ajustar parâmetros de pós-processamento (k=5, min_duration=10s)")
print("  2. Adicionar features de contexto temporal (média móvel de prob)")
print("  3. Testar threshold mais baixo (0.25-0.35)")
print("  4. Calibração de probabilidades (Platt scaling)")
print("  5. Ensemble voting entre modelos")

print("\n" + "="*80)
print("✅ PIPELINE V2 CONCLUÍDO COM SUCESSO!")
print("="*80)


SUMÁRIO FINAL - BASELINE vs V2

1. MÉTRICAS GERAIS (média de todas as configurações)
--------------------------------------------------------------------------------

SENSITIVITY:
  Baseline: 0.522
  V2: 0.509
  Melhoria: -0.013 (-2.5%)

PRECISION:
  Baseline: 0.639
  V2: 0.045
  Melhoria: -0.594 (-92.9%)

AUC_PR:
  Baseline: 0.439
  V2: 0.144
  Melhoria: -0.295 (-67.2%)

FAR_PER_HOUR:
  Baseline: 77.95
  V2: 8.89
  Redução: 69.06 (88.6%)


2. TOP 3 CONFIGURAÇÕES - BASELINE
--------------------------------------------------------------------------------


,FS,Janela,Ratio,Modelo,sensitivity,far_per_hour
0,FS-A,4s_50,1:3,XGB,0.577975,118.584503
1,FS-A,5s_50,1:3,RF,0.573872,99.477346
2,FS-A,4s_50,1:3,RF,0.572955,123.438281



2. TOP 3 CONFIGURAÇÕES - V2
--------------------------------------------------------------------------------


,FS,Janela,Ratio,Modelo,sensitivity,far_per_hour
0,FS-A,6s_50,1:3,RF,0.626389,9.748436
1,FS-A,4s_50,1:3,XGB,0.615278,12.253852
2,FS-A,6s_50,1:3,SVM,0.558333,7.884449




3. ARQUIVOS SALVOS
--------------------------------------------------------------------------------
  • results_baseline_all.csv
  • ranking_baseline.csv
  • results_v2_all.csv
  • ranking_v2.csv
  • comparison_baseline_vs_v2.csv
  • threshold_analysis.csv


4. PRÓXIMOS PASSOS (se FAR ainda > 15/h)
--------------------------------------------------------------------------------
  1. Ajustar parâmetros de pós-processamento (k=5, min_duration=10s)
  2. Adicionar features de contexto temporal (média móvel de prob)
  3. Testar threshold mais baixo (0.25-0.35)
  4. Calibração de probabilidades (Platt scaling)
  5. Ensemble voting entre modelos

✅ PIPELINE V2 CONCLUÍDO COM SUCESSO!
